In [ ]:
from LSTM_model import LSTMModel
from Transformer_model import TransformerModel
from GRU_model import GRUModel
from CNN_model import CNNModel
from LightGBM_model import LightGBMModel
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [ ]:
FEATURE_COLS = ['open', 'high', 'low', 'close', 'volume', 'is_day_start', 'is_day_end']
SEQ_LEN      = 30
SPLIT        = (0.70, 0.85)   # train / val / test

def prepare(df: pd.DataFrame):
    """
    Преобразует df в (X_train, y_train, X_val, y_val, X_test, y_test).
    Таргет: лог-доходность следующей свечи (регрессия).
    """
    df = df.copy()
    df['target'] = np.log(df['close'].shift(-1) / df['close'])
    df = df.dropna()

    X = df[FEATURE_COLS].values
    y = df['target'].values

    n = len(X)
    t1, t2 = int(n * SPLIT[0]), int(n * SPLIT[1])

    X_tr, y_tr = X[:t1],    y[:t1]
    X_vl, y_vl = X[t1:t2],  y[t1:t2]
    X_te, y_te = X[t2:],    y[t2:]

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_vl = scaler.transform(X_vl)
    X_te = scaler.transform(X_te)

    return X_tr, y_tr, X_vl, y_vl, X_te, y_te

# ── загрузка и препроцессинг ──────────────────────────────────────────────────
tickers = {"ABIO": "ABIO.csv", "ABRD": "ABRD.csv", "AFKS": "AFKS.csv"}

data = {}   # ticker -> (X_tr, y_tr, X_vl, y_vl, X_te, y_te)
for name, path in tickers.items():
    data[name] = prepare(pd.read_csv(path))

# удобные алиасы для single-режима (первый тикер)
X_train, y_train, X_val, y_val, X_test, y_test = data["ABIO"]

INPUT_SIZE   = len(FEATURE_COLS)
NUM_TICKERS  = len(tickers)
ticker_to_id = {t: i for i, t in enumerate(tickers)}

print("Shapes (ABIO):", X_train.shape, X_val.shape, X_test.shape)
print("Ticker → id:", ticker_to_id)

In [ ]:
# ── pooled: объединяем все тикеры в один массив ───────────────────────────────
def make_pooled(split: str):
    """split: 'train' | 'val' | 'test'"""
    idx = {"train": 0, "val": 2, "test": 4}[split]
    X_parts, y_parts, ids_parts = [], [], []
    for ticker, arrays in data.items():
        X_s, y_s = arrays[idx], arrays[idx + 1]
        X_parts.append(X_s)
        y_parts.append(y_s)
        ids_parts.append(np.full(len(X_s), ticker_to_id[ticker], dtype=np.int64))
    return np.concatenate(X_parts), np.concatenate(y_parts), np.concatenate(ids_parts)

Xp_tr, yp_tr, ids_tr = make_pooled("train")
Xp_vl, yp_vl, ids_vl = make_pooled("val")
Xp_te, yp_te, ids_te = make_pooled("test")

# словари для scores_per_ticker
X_test_dict = {t: data[t][4] for t in tickers}
y_test_dict  = {t: data[t][5] for t in tickers}

print("Pooled train:", Xp_tr.shape, "val:", Xp_vl.shape, "test:", Xp_te.shape)

In [ ]:
# ── helpers ───────────────────────────────────────────────────────────────────
FIT_KW   = dict(epochs=30, verbose=True)   # общие параметры обучения
SCORES_KW = {}                             # можно добавить общие параметры scores

def run_single(model, name):
    """Обучение и оценка в режиме single (один тикер — ABIO)."""
    print(f"\n{'='*60}\n  {name}  |  mode=single\n{'='*60}")
    model.fit(X_train, y_train, X_val=X_val, y_val=y_val, **FIT_KW)
    model.plot_loss()
    print(model.scores(X_test, y_test, name))
    print("predict_last:", model.predict_last(X_test))

def run_pooled(model, name):
    """Обучение и оценка в режиме pooled (все тикеры)."""
    print(f"\n{'='*60}\n  {name}  |  mode=pooled\n{'='*60}")
    model.fit(Xp_tr, yp_tr, ticker_ids=ids_tr,
              X_val=Xp_vl, y_val=yp_vl, ticker_ids_val=ids_vl, **FIT_KW)
    model.plot_loss()
    print(model.scores_per_ticker(X_test_dict, y_test_dict, ticker_to_id))

In [ ]:
# ── single mode: все 5 моделей ────────────────────────────────────────────────
models_single = [
    (LSTMModel(        input_size=INPUT_SIZE, hidden_size=64, num_layers=2,   seq_len=SEQ_LEN, mode="single"), "LSTM"),
    (GRUModel(         input_size=INPUT_SIZE, hidden_size=64, num_layers=2,   seq_len=SEQ_LEN, mode="single"), "GRU"),
    (CNNModel(         input_size=INPUT_SIZE, num_filters=64, num_layers=2,   seq_len=SEQ_LEN, mode="single"), "CNN"),
    (TransformerModel( input_size=INPUT_SIZE, d_model=64,     nhead=4,        seq_len=SEQ_LEN, mode="single"), "Transformer"),
    (LightGBMModel(    input_size=INPUT_SIZE,                                 seq_len=SEQ_LEN, mode="single"), "LightGBM"),
]

for model, name in models_single:
    run_single(model, name)

In [ ]:
# ── pooled mode: все 5 моделей ────────────────────────────────────────────────
models_pooled = [
    (LSTMModel(        input_size=INPUT_SIZE, hidden_size=64, num_layers=2,   seq_len=SEQ_LEN, mode="pooled", num_tickers=NUM_TICKERS), "LSTM"),
    (GRUModel(         input_size=INPUT_SIZE, hidden_size=64, num_layers=2,   seq_len=SEQ_LEN, mode="pooled", num_tickers=NUM_TICKERS), "GRU"),
    (CNNModel(         input_size=INPUT_SIZE, num_filters=64, num_layers=2,   seq_len=SEQ_LEN, mode="pooled", num_tickers=NUM_TICKERS), "CNN"),
    (TransformerModel( input_size=INPUT_SIZE, d_model=64,     nhead=4,        seq_len=SEQ_LEN, mode="pooled", num_tickers=NUM_TICKERS), "Transformer"),
    (LightGBMModel(    input_size=INPUT_SIZE,                                 seq_len=SEQ_LEN, mode="pooled"),                          "LightGBM"),
]

for model, name in models_pooled:
    run_pooled(model, name)

In [ ]:
# ── finetune mode: pretrain на всех → finetune на ABIO ────────────────────────
def run_finetune(model, name, ticker="ABIO", ft_epochs=10):
    print(f"\n{'='*60}\n  {name}  |  mode=finetune  →  ticker={ticker}\n{'='*60}")
    tid = ticker_to_id[ticker]
    Xf_tr, yf_tr, Xf_vl, yf_vl, Xf_te, yf_te = data[ticker]

    # pretrain на всех тикерах
    model.pretrain(Xp_tr, yp_tr, ticker_ids=ids_tr,
                   X_val=Xp_vl, y_val=yp_vl, ticker_ids_val=ids_vl,
                   epochs=30, verbose=False)

    # finetune на одном тикере
    if isinstance(model, LightGBMModel):
        model.finetune(Xf_tr, yf_tr, ticker_id=tid,
                       X_val=Xf_vl, y_val=yf_vl,
                       n_finetune_estimators=50, verbose=True)
        ids_te_one = np.full(len(Xf_te), tid, dtype=np.int64)
        print(model.scores(Xf_te, yf_te, name, ticker_ids=ids_te_one, use_finetune=True))
    else:
        model.finetune(Xf_tr, yf_tr, ticker_id=tid,
                       X_val=Xf_vl, y_val=yf_vl,
                       epochs=ft_epochs, verbose=True)
        model.plot_loss()
        ids_te_one = np.full(len(Xf_te), tid, dtype=np.int64)
        print(model.scores(Xf_te, yf_te, name, ticker_ids=ids_te_one))

models_finetune = [
    (LSTMModel(        input_size=INPUT_SIZE, hidden_size=64, num_layers=2, seq_len=SEQ_LEN, mode="finetune", num_tickers=NUM_TICKERS), "LSTM"),
    (GRUModel(         input_size=INPUT_SIZE, hidden_size=64, num_layers=2, seq_len=SEQ_LEN, mode="finetune", num_tickers=NUM_TICKERS), "GRU"),
    (CNNModel(         input_size=INPUT_SIZE, num_filters=64, num_layers=2, seq_len=SEQ_LEN, mode="finetune", num_tickers=NUM_TICKERS), "CNN"),
    (TransformerModel( input_size=INPUT_SIZE, d_model=64,     nhead=4,      seq_len=SEQ_LEN, mode="finetune", num_tickers=NUM_TICKERS), "Transformer"),
    (LightGBMModel(    input_size=INPUT_SIZE,                               seq_len=SEQ_LEN, mode="finetune"),                          "LightGBM"),
]

for model, name in models_finetune:
    run_finetune(model, name)

In [ ]:
# ── итоговая таблица метрик по всем single-моделям ───────────────────────────
import pandas as pd

results = []
for model, name in models_single:
    row = model.scores(X_test, y_test, name)
    results.append(row)

pd.concat(results).sort_values("dir_accuracy", ascending=False)
